## Objetivo

Demonstrar que toda imagem digital pode ser representada como uma **matriz numérica**, onde cada elemento contém os valores de cor (R, G, B) de um pixel.

## Metodologia

1. Carregar uma imagem e redimensioná-la para 50×50 pixels
2. Extrair cada pixel como uma tupla (Y, X, R, G, B)
3. Exportar esses dados para um arquivo CSV (planilha)
4. Reconstruir a imagem **exclusivamente** a partir do CSV
5. Comparar visualmente o resultado — provando que nenhuma informação foi perdida

Se a reconstrução for idêntica, fica provado que a imagem **é** a matriz de números.

## Passo 1 — Importação das Bibliotecas

Utilizamos:
- **Pillow (PIL)**: para manipulação de imagens (carregar, redimensionar, criar)
- **pandas**: para estruturar os dados dos pixels em formato tabular
- **NumPy**: para criar arrays numéricos eficientes na reconstrução
- **pathlib**: para manipulação de caminhos de arquivos
- **IPython.display**: para exibir as imagens diretamente no notebook

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

print("Bibliotecas importadas com sucesso.")

## Passo 2 — Definição dos Caminhos

Definimos os diretórios e caminhos de saída. A imagem de entrada deve estar na pasta `images/`.

In [ ]:
NOTEBOOK_DIR = Path(".").resolve()
IMAGES_DIR = NOTEBOOK_DIR / "images"
CSV_PATH = NOTEBOOK_DIR / "pixel_data.csv"
RECONSTRUCTED_PATH = NOTEBOOK_DIR / "reconstructed_image.png"

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

print(f"Diretório de imagens: {IMAGES_DIR}")
print(f"CSV de saída: {CSV_PATH}")
print(f"Imagem reconstruída: {RECONSTRUCTED_PATH}")

## Passo 3 — Carregar e Redimensionar a Imagem

Buscamos automaticamente a primeira imagem disponível na pasta `images/` e a redimensionamos para **50×50 pixels**.

O redimensionamento garante que a matriz resultante tenha um tamanho controlado (2.500 pixels = 2.500 linhas no CSV), facilitando a visualização e análise dos dados.

Também convertemos para o modo **RGB** (3 canais de cor), garantindo uniformidade independente do formato original.

In [ ]:
# Buscar a primeira imagem no diretório
image_path = None
for file in sorted(IMAGES_DIR.iterdir()):
    if file.suffix.lower() in SUPPORTED_EXTENSIONS:
        image_path = file
        break

if image_path is None:
    raise FileNotFoundError(
        "Nenhuma imagem encontrada na pasta 'images/'. "
        "Coloque um arquivo .png, .jpg, .jpeg ou .bmp lá e execute novamente."
    )

print(f"Imagem encontrada: {image_path.name}")

# Carregar e redimensionar
original = Image.open(image_path).convert("RGB")
print(f"Tamanho original: {original.size[0]}x{original.size[1]} pixels")

image = original.resize((50, 50))
print(f"Tamanho redimensionado: {image.size[0]}x{image.size[1]} pixels")

print("\nImagem redimensionada:")
display(image.resize((200, 200), Image.NEAREST))  # Ampliada para visualização

## Passo 4 — Extrair a Matriz de Pixels

Aqui está o núcleo da prova. Percorremos **cada pixel** da imagem, linha por linha (Y) e coluna por coluna (X), e extraímos seus valores de cor:

- **Y**: posição vertical (linha da matriz)
- **X**: posição horizontal (coluna da matriz)
- **R**: componente vermelho (0–255)
- **G**: componente verde (0–255)
- **B**: componente azul (0–255)

Cada pixel se torna uma linha no DataFrame. Uma imagem 50×50 gera exatamente **2.500 linhas**.

Isso demonstra que a imagem é, literalmente, uma tabela de números organizados em formato matricial.

In [ ]:
width, height = image.size
rows = []

for y in range(height):
    for x in range(width):
        r, g, b = image.getpixel((x, y))
        rows.append({"Y": y, "X": x, "R": r, "G": g, "B": b})

df = pd.DataFrame(rows)

print(f"Total de pixels extraídos: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print(f"\nPrimeiros 10 pixels (canto superior esquerdo):")
df.head(10)

### Estatísticas da Matriz

Podemos observar as estatísticas descritivas dos valores de cor, confirmando que todos estão no intervalo [0, 255].

In [ ]:
df[["R", "G", "B"]].describe()

## Passo 5 — Exportar para CSV (Planilha)

Salvamos o DataFrame como um arquivo CSV. Este arquivo contém **toda** a informação numérica da imagem.

O CSV pode ser aberto em qualquer editor de planilhas (Excel, Google Sheets, LibreOffice) para verificar que a imagem é, de fato, apenas uma tabela de números.

In [ ]:
df.to_csv(CSV_PATH, index=False)
print(f"Dados exportados para: {CSV_PATH.name}")
print(f"Tamanho do arquivo: {CSV_PATH.stat().st_size:,} bytes")
print(f"Linhas (pixels): {len(df)}")
print(f"Colunas: {list(df.columns)}")

## Passo 6 — Reconstruir a Imagem a Partir do CSV

Agora, **ignoramos completamente a imagem original**. Lemos apenas o arquivo CSV e reconstruímos a imagem pixel por pixel, usando somente os valores numéricos da tabela.

O processo:
1. Ler o CSV e determinar as dimensões da imagem (max(Y)+1 × max(X)+1)
2. Criar uma imagem em branco com essas dimensões
3. Para cada linha do CSV, definir o pixel na posição (X, Y) com a cor (R, G, B)

Se a imagem resultante for **idêntica** à original redimensionada, fica provado que **a matriz numérica contém 100% da informação da imagem**.

In [ ]:
# Ler o CSV (ignorando a imagem original)
df_csv = pd.read_csv(CSV_PATH).astype(int)

# Determinar dimensões
csv_height = df_csv["Y"].max() + 1
csv_width = df_csv["X"].max() + 1
print(f"Dimensões detectadas no CSV: {csv_width}x{csv_height} pixels")

# Criar array de pixels e preencher
pixels = np.zeros((csv_height, csv_width, 3), dtype=np.uint8)
for _, row in df_csv.iterrows():
    pixels[row["Y"], row["X"]] = [row["R"], row["G"], row["B"]]

# Criar imagem reconstruída
reconstructed = Image.fromarray(pixels)

# Salvar
reconstructed.save(RECONSTRUCTED_PATH)
print(f"Imagem reconstruída salva em: {RECONSTRUCTED_PATH.name}")

print("\nImagem reconstruída:")
display(reconstructed.resize((200, 200), Image.NEAREST))  # Ampliada para visualização

## Passo 7 — Comparação Visual

Exibimos lado a lado a imagem original (redimensionada) e a imagem reconstruída a partir do CSV para confirmar visualmente que são idênticas.

In [ ]:
# Comparação lado a lado
scale = 200
original_scaled = image.resize((scale, scale), Image.NEAREST)
reconstructed_scaled = reconstructed.resize((scale, scale), Image.NEAREST)

# Criar imagem de comparação
comparison = Image.new("RGB", (scale * 2 + 20, scale + 40), (255, 255, 255))
comparison.paste(original_scaled, (0, 40))
comparison.paste(reconstructed_scaled, (scale + 20, 40))

print("Original (esquerda) vs. Reconstruída do CSV (direita):")
display(comparison)

# Verificação numérica
original_array = np.array(image)
reconstructed_array = np.array(reconstructed)
are_identical = np.array_equal(original_array, reconstructed_array)

print(f"\nAs imagens são numericamente idênticas? {'SIM' if are_identical else 'NÃO'}")
if are_identical:
    print("\nConclusão: A imagem é inteiramente representada por sua matriz numérica.")
    print("Nenhuma informação foi perdida na conversão para números e vice-versa.")

## Conclusão

Demonstramos que:

1. Uma imagem digital de dimensão **H × W** é equivalente a uma **matriz H × W**, onde cada elemento é um vetor de 3 componentes (R, G, B)
2. Essa matriz pode ser serializada como uma tabela de números (CSV) sem perda de informação
3. A imagem pode ser perfeitamente reconstruída a partir da tabela numérica, comprovando que a informação visual está integralmente codificada na estrutura matricial

Portanto, **uma imagem digital é, por definição, uma matriz de números**, e a Álgebra Linear é a linguagem matemática subjacente à representação de dados visuais em sistemas computacionais.